# 📋 TV4: Trạng thái & Bàn giao (Frontend, Security QA, Báo cáo)

Notebook này trả lời 3 câu hỏi cho cả nhóm:
1. **TV4 đã làm xong gì?** Có cell chạy kiểm chứng ngay, không cần tin lời.
2. **Còn thiếu gì?** Những thứ chưa xong, không được trình bày như đã xong.
3. **Mỗi thành viên cần làm gì tiếp?** Checklist theo TV1 / TV2 / TV3.

▶️ Chạy **Run All** để cập nhật kết quả kiểm chứng (cần Python + `pip install -r tests/security/requirements.txt`).

---
## 1. ✅ Đã xong

| Hạng mục | Vị trí | Kiểm chứng |
|---|---|---|
| Web App: Đăng nhập Entra/OIDC (MSAL, PKCE), Chuyển Org, Dự án, Task Board, Quản lý Member, Audit History, Export CSV + tải qua SAS | `frontend/` | Cell 1.2 (type-check) |
| Automated API Security Tests 1–3 (pytest) | `tests/security/` | Cell 1.1 (selfcheck) |
| Ghi bằng chứng: input, timestamp, raw HTTP, tự che JWT/SAS/password | `tests/security/evidence.py` | `test_0_evidence_redaction.py` (trong 1.1) |
| Máy quét rò rỉ bí mật trước khi nộp | `tests/security/scan_secrets.py` | Cell 1.3 |
| Mock API theo contract (chỉ dùng cho dev/QA) | `tests/mock_api/server.py` | `API_CONTRACT_PROPOSAL.ipynb` phần B |
| Bản nháp Design Report | `docs/tv4/DESIGN_REPORT.ipynb` | Có chỗ `⟦ĐIỀN⟧` chờ số liệu thật |
| Kịch bản Demo 10–15 phút | `docs/tv4/DEMO_SCRIPT.ipynb` | — |
| API Contract đề xuất | `docs/tv4/API_CONTRACT_PROPOSAL.ipynb` | **Chờ TV2 xác nhận** |

In [1]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists() or (p / "infra").is_dir())
print("Repo root:", ROOT.name)

Repo root: multi-tenant-project-management


### 1.1 Selfcheck: bộ test có thực sự bắt được lỗi không?

Chạy bộ test 5 lần trên mock API:
- 1 lần với mock **an toàn** → tất cả phải **PASS**;
- 4 lần, mỗi lần mock **cố tình có 1 lỗ hổng** → test tương ứng phải **FAIL**.

| `--vuln` | Lỗ hổng giả lập | Test phải bắt |
|---|---|---|
| `idor` | API không kiểm tra tài nguyên thuộc tenant nào | Test 1 |
| `stale-membership` | API vẫn cho user đã bị thu hồi truy cập | Test 2 |
| `global-unique` | `UNIQUE(name)` toàn cục thay vì `(tenant_id, name)` | Test 3 |
| `shared-cache` | Cache key không có `tenant_id` | Test 3 |

In [2]:
import subprocess, sys, os
res = subprocess.run([sys.executable, str(ROOT / "tests" / "security" / "selfcheck.py")],
                     capture_output=True, text=True, encoding="utf-8", env={**os.environ, "PYTHONIOENCODING": "utf-8"})
print(res.stdout.strip() or res.stderr[-2000:])

[OK ] secure mock        -> tests must PASS : 34 passed in 1.94s
[OK ] --vuln idor             -> test_1_cross_tenant_idor.py must FAIL : 15 failed, 12 passed in 1.04s
[OK ] --vuln stale-membership -> test_2_membership_revocation.py must FAIL : 1 failed, 1 passed in 0.36s
[OK ] --vuln global-unique    -> test_3_duplicate_name_cache.py must FAIL : 1 failed in 0.27s
[OK ] --vuln shared-cache     -> test_3_duplicate_name_cache.py must FAIL : 1 failed in 0.32s

Selfcheck: 5/5 scenarios behaved as expected


### 1.2 Frontend type-check
Cần `npm ci` trong `frontend/` trước. Nếu chưa cài, cell sẽ bỏ qua.

In [3]:
import shutil
fe = ROOT / "frontend"
npx = shutil.which("npx")
if not (fe / "node_modules").exists() or not npx:
    print("⏸️ Bỏ qua: chạy `cd frontend && npm ci` trước")
else:
    r = subprocess.run([npx, "tsc", "-b", "--noEmit"], cwd=fe, capture_output=True, text=True, encoding="utf-8")
    print("✅ TypeScript strict: 0 lỗi" if r.returncode == 0 else r.stdout[-2000:])
    for f in fe.glob("*.tsbuildinfo"):
        f.unlink()

✅ TypeScript strict: 0 lỗi


### 1.3 Quét bí mật trong code TV4 (JWT, SAS, AccountKey, password)

In [4]:
targets = [ROOT / "frontend" / "src", ROOT / "frontend" / ".env.example", ROOT / "docs" / "tv4", ROOT / "tests"]
r = subprocess.run([sys.executable, str(ROOT / "tests" / "security" / "scan_secrets.py"), *map(str, targets)],
                   capture_output=True, text=True, encoding="utf-8", env={**os.environ, "PYTHONIOENCODING": "utf-8"})
print(("✅ " if r.returncode == 0 else "❌ ") + r.stdout.strip().splitlines()[-1].split(":")[-1].strip())

✅ 0 nghi vấn


---
## 2. ⏳ Chưa làm được (phụ thuộc thành viên khác)

| Việc | Vì sao chưa | Khi nào |
|---|---|---|
| Chạy Test 1–3 trên **backend thật** | Repo chưa có API Spring Boot của TV2. Mọi kết quả PASS hiện tại là trên **mock** | Khi TV2 merge API |
| Đăng nhập Entra **thật** | Cần tenant External ID + 2 app registration (SPA + API) | Tuần 3–4 |
| Bằng chứng Azure trong `docs/tv4/evidence/` | Chỉ thu được khi bật Azure | Đợt 2, Tuần 6 |
| Kiểm tra UI bằng mắt trên trình duyệt | Phiên phát triển chỉ kiểm qua build, type-check và gọi HTTP | Demo nội bộ cuối Tuần 4 |

---
## 3. 🤝 Cần thành viên khác xử lý

### TV2: Backend
- [ ] Xác nhận hoặc sửa `API_CONTRACT_PROPOSAL.ipynb` (5 điểm ở phần C). Khi đổi, TV4 chỉ sửa `frontend/src/api/client.ts` và `tests/security/api_paths.py`.
- [ ] Bật **CORS** cho origin Static Web App và `http://localhost:5173`. Allow `Authorization, Content-Type, X-Request-Id`; expose `X-Request-Id`.
- [ ] Nhận `X-Request-Id` từ client (nếu hợp lệ) → MDC + `audit_events.request_id`, để bằng chứng test nối được với log.
- [ ] Seed script có: Org A gồm ≥ 1 OWNER, 1 GUEST, 1 MEMBER "User X"; Org B gồm 1 OWNER + project + task; 1 user thuộc 2 org (demo Chuyển Org).

### TV1: Infra / CI-CD
- [ ] Thêm `azurerm_static_web_app` (SKU Free) + output `default_host_name` → thêm vào CORS của API.
- [ ] Workflow frontend: `npm ci && npm run build` trong `frontend/`, truyền `VITE_API_BASE_URL`, `VITE_ENTRA_CLIENT_ID`, `VITE_ENTRA_AUTHORITY`, `VITE_API_SCOPE` (không phải secret, đặt ở GitHub Variables).
- [ ] Job CI chạy `python tests/security/selfcheck.py` (không cần Azure, ~10 giây) để chặn PR làm hỏng bộ test.
- [ ] Cân nhắc `public_network_access_enabled = false` / network rules cho Storage Account.

### TV3: Worker / Observability
- [ ] Chốt với TV2 tên trạng thái export (`PENDING` hay `QUEUED`, có `FAILED` không) — xem `API_CONTRACT_PROPOSAL.ipynb` phần C, điểm 6.
- [ ] k6: đổi `API_PATH` mặc định `/api/v1/projects` thành `/api/v1/orgs/{tenantId}/projects` theo contract.
- [ ] Thêm alert trên **metric tenant-mismatch** của Worker (hiện chỉ có `Http5xx` của API).
- [ ] Gửi TV4: ảnh Dashboard, email Alert, KQL truy vết `jobId`, p95 k6 → điền Design Report §4, §6.

Cell dưới **tự dò Terraform** để biết các việc của TV1/TV3 ở trên đã xong chưa:

In [5]:
import re
tf = "\n".join(p.read_text(encoding="utf-8") for p in (ROOT / "infra").glob("*.tf"))
storage = (ROOT / "infra" / "storage_queue.tf").read_text(encoding="utf-8")
gaps = [
    ("TV1", "Static Web App cho frontend",            bool(re.search(r'resource\s+"azurerm_static_web_app"', tf))),
    ("TV1", "Storage tắt public network access",      bool(re.search(r"public_network_access_enabled\s*=\s*false", storage))),
    ("TV2/TV1", "CORS cho API (app_service site_config.cors)", "cors" in tf.lower()),
    ("TV3/TV1", "Alert không chỉ dựa trên Http5xx (tenant-mismatch)",
        len(re.findall(r'resource\s+"azurerm_monitor_(metric|scheduled_query_rules)_alert', tf)) > 1
        or "mismatch" in tf.lower()),
]
for owner, item, done in gaps:
    print(f"{'✅ xong ' if done else '⏳ chưa '} [{owner:7}] {item}")

⏳ chưa  [TV1    ] Static Web App cho frontend
⏳ chưa  [TV1    ] Storage tắt public network access
⏳ chưa  [TV2/TV1] CORS cho API (app_service site_config.cors)
⏳ chưa  [TV3/TV1] Alert không chỉ dựa trên Http5xx (tenant-mismatch)


---
## 4. 🗓️ Kế hoạch Đợt 2 (Tuần 6) của TV4

| Bước | Lệnh / Việc |
|---|---|
| 1 | Lấy token mới cho 4 user (xem `tests/security/README.md`) → `tests/security/tokens.local.json` |
| 2 | `.\tests\security\run.ps1 -Target azure -BaseUrl <api_https_url>` |
| 3 | `python tests/security/scan_secrets.py tests/security/evidence` → copy bản sạch vào `docs/tv4/evidence/<run>/` → PR |
| 4 | Chụp UI: Guest bị chặn, User X bị 403 sau khi thu hồi, Export tải qua SAS |
| 5 | Điền các chỗ `⟦ĐIỀN⟧` trong `DESIGN_REPORT.ipynb` (§4 sẽ tự đọc kết quả từ `docs/tv4/evidence/`) |